In [1]:
import pandas as pd
import sqlite3
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Zenginleştirilmiş Veriyi Veritabanından Çekiyoruz
conn = sqlite3.connect('../app.db')
# NLP yapabilmemiz için açıklaması (description) olan filmleri alıyoruz
movies_df = pd.read_sql_query("SELECT id, title, description, genres FROM movies WHERE description IS NOT NULL", conn)
conn.close()

# 2. Doğal Dil İşleme (NLP) Hazırlığı
# Makinenin okuyabilmesi için filmin Türünü (genres) ve Özetini (description) tek bir uzun metin (content) yapıyoruz.
movies_df['content'] = movies_df['genres'].fillna('') + " " + movies_df['description'].fillna('')

print("TF-IDF İçin Hazırlanan Ham Metinler (İlk 3 Film):")
display(movies_df[['title', 'content']].head(3))

# 3. TF-IDF Vektörizasyonu (PROPOZAL KANITI)
# 'stop_words' ile 'and, the, is' gibi gereksiz kelimeleri çöpe atıp, metinleri 0 ve 1'lerden oluşan matrislere çeviriyoruz.
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(movies_df['content'])

print(f"\n--- NLP İŞLEMİ (TF-IDF) BAŞARILI ---")
print(f"Oluşan Dev Matrisin Boyutu (Shape): {tfidf_matrix.shape}")
print("(Anlamı: Satırlar Filmleri, Sütunlar ise Kelime Köklerini temsil eder)\n")

# 4. Kosinüs Benzerliği (Cosine Similarity)
# İki filmin vektör açısını (Dot Product) hesaplayarak birbirlerine olan matematiksel benzerlik oranlarını buluyoruz.
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Kosinüs Benzerlik Matrisi Başarıyla Hesaplandı: {cosine_sim.shape}")

TF-IDF İçin Hazırlanan Ham Metinler (İlk 3 Film):


,title,content
0,Inception,Science Fiction Action Rüyalar içinde rüya gör...
1,The Shawshank Redemption,Drama Haksız yere ömür boyu hapse mahkum edile...
2,The Dark Knight,"Action Crime Drama Batman, Gotham şehrini terö..."



--- NLP İŞLEMİ (TF-IDF) BAŞARILI ---
Oluşan Dev Matrisin Boyutu (Shape): (13, 322)
(Anlamı: Satırlar Filmleri, Sütunlar ise Kelime Köklerini temsil eder)

Kosinüs Benzerlik Matrisi Başarıyla Hesaplandı: (13, 13)


In [2]:
# 5. İçerik Tabanlı Modelin Test Edilmesi (Propozal Kanıtı)
# Film başlıklarını kolayca bulmak için bir fihrist (sözlük) oluşturuyoruz
indeksler = pd.Series(movies_df.index, index=movies_df['title']).drop_duplicates()

def icerik_tabanli_oneri_ver(film_adi, kosinus_matrisi=cosine_sim):
    if film_adi not in indeksler:
        return "Film bulunamadı."
        
    idx = indeksler[film_adi]

    # Bu filmin diğer 12 filmle olan açısal benzerlik skorlarını (0 ile 1 arası) alıyoruz
    benzerlik_skorlari = list(enumerate(kosinus_matrisi[idx]))

    # En çok benzeyenden (1'e en yakın olan) en az benzeyene doğru sıralıyoruz
    benzerlik_skorlari = sorted(benzerlik_skorlari, key=lambda x: x[1], reverse=True)

    # Kendisi hariç (kendisine benzerliği %100'dür), en çok benzeyen 3 filmi seçiyoruz
    en_benzer_3_film = benzerlik_skorlari[1:4]

    print(f"🎯 '{film_adi}' Filmi İçin Yapay Zeka Önerileri:")
    print("-" * 50)
    for i, skor in en_benzer_3_film:
        onerilen_film = movies_df['title'].iloc[i]
        yuzde = round(skor * 100, 1)
        print(f"> {onerilen_film} (Benzerlik Oranı: %{yuzde})")

# TEST EDİYORUZ: Inception filmine hangi 3 filmi önerecek?
icerik_tabanli_oneri_ver('Inception')

🎯 'Inception' Filmi İçin Yapay Zeka Önerileri:
--------------------------------------------------
> The Dark Knight (Benzerlik Oranı: %4.5)
> Maleficent (Benzerlik Oranı: %4.4)
> Avatar: Ateş ve Kül (Benzerlik Oranı: %2.2)
